In [1]:
# The CSM class will allow us to find and use cost and scaling models
from csm import CSM

# We can see what models are available to us (in the default model location)
available_models = CSM.get_available_models()
print(available_models.keys())

dict_keys(['Empirical2020', 'Empirical2024Offshore', 'Empirical2024Onshore', 'Empirical2030', 'Replicate2015', 'Replicate2015Excel'])


In [2]:
# Let's use the 'Empirical2020' model. We can create an instance of it using the
# CSM.from_name method
model = CSM.from_name("Empirical2020")

# We can also specify the path to look for models if required
model = CSM.from_name("Empirical2020", "./csm/model")

# Alternatively we can use the CSM class directly (if we are using the default model
# directory)
model = CSM("Empirical2020")

print(model)

CSM(Empirical2020)


In [3]:
# Let's see what this model can do

# Get all the possible outputs of the model
print(model.get_outputs())

('bedplate_mass', 'bedplate_cost', 'rotor_radius', 'blade_mass', 'blade_cost', 'blade_surface_area', 'blade_transport_cost', 'braking_system_mass', 'braking_system_cost', 'controls_cost', 'railing_platform_mass', 'crane_mass', 'crane_cost', 'electrical_connection_cost', 'rotor_angular_velocity_max', 'rotor_torque_max', 'gearbox_mass', 'gearbox_cost', 'generator_mass', 'generator_cost', 'hub_mass', 'hub_cost', 'hub_surface_area', 'hub_transport_cost', 'hvac_mass', 'hvac_cost', 'low_speed_shaft_mass', 'low_speed_shaft_cost', 'main_bearing_mass', 'main_bearing_cost', 'transformer_power_electronics_mass', 'transformer_power_electronics_cost', 'yaw_system_mass', 'yaw_system_cost', 'railing_platform_cost', 'nacelle_cover_mass', 'nacelle_cover_cost', 'nacelle_cost', 'nacelle_mass', 'nacelle_drivetrain_transport_cost', 'nacelle_length', 'nacelle_lift_height', 'nacelle_power_electronics_cost', 'nacelle_surface_area', 'nose_cone_mass', 'nose_cone_cost', 'swept_area', 'tower_mass', 'num_tower_sec

In [4]:
# Get all the required inputs to the model. If we know all these parameters, we can
# calculate all of the above outputs
print(model.get_inputs())

('BOS_cost', 'hub_height', 'num_bearings', 'num_blades', 'rotor_diameter', 'rotor_efficiency_max', 'tip_speed_max', 'turbine_rating_MW')


In [5]:
# Let's focus on a particular output, blade_cost. We can see the function code itself if
# we want to
model.display_function("blade_cost")

    def blade_cost(blade_mass):
        return 15.9432 * blade_mass



In [6]:
# If we are just interested in which variables depend on each other we can use
# get_function_args to get a dictionary of which function names use which function
# arguments
function_args = model.get_function_args()
print(function_args["blade_cost"])

['blade_mass']


In [7]:
# So if we know the blade_mass we can calculate the blade_cost. But blade_mass is also
# defined by a function. It might be easier to figure out which of the inputs are needed
# to calculate blade_cost, rather than intermediate calculation steps like blade_mass
print(model.required_inputs_to_calculate("blade_cost"))

['rotor_diameter']


In [8]:
# So using this model we can estimate the blade cost based only on the rotor_diameter

# Let's create a dict to contain our required inputs (so far just the rotor_diameter)
inputs = {
    "rotor_diameter": 162.0,
}

# We can calculate the blade cost using the inputs dict either as-is or by unpacking it
print(model.calculate("blade_cost", inputs))
print(model.calculate("blade_cost", **inputs))

347627.7901838095
347627.7901838095


In [9]:
# If we provide inputs that are not relevant to the model they will be ignored silently
print(model.calculate("blade_cost", rotor_diameter=162, irrelevant_parameter=100))

347627.7901838095


In [10]:
# Just because it might be easier to use only input values doesn't mean we have to. We
# know from above that we can calculate blade_cost given blade_mass.
print(model.calculate("blade_cost", blade_mass=20_000))

# This value is different to the above because by specifying our own value of
# blade_mass, we are not using the same relationship between blade_mass and
# rotor_diameter that the model does.

318864.0


In [11]:
# Let's try specifying both blade_mass and rotor_diameter directly, and intentionally
# provide values that do not agree with each other
print(model.calculate("blade_cost", blade_mass=20_000, rotor_diameter=162))
print(model.calculate("blade_cost", rotor_diameter=162, blade_mass=20_000))

# It doesn't matter what order we use, the model will always choose the parameter that
# requires less effort to perform the calculation (in this case it uses blade_mass and
# it does not use rotor_diameter)

318864.0
318864.0


In [12]:
# If we respect the relationship between blade_mass and rotor_diameter, the values will
# be consistent with each other

# First, calculate the blade_mass using our inputs dict (which right now contains just
# rotor_diameter)
model_blade_mass = model.calculate("blade_mass", inputs)
print("model_blade_mass", model_blade_mass)

# Using this calculated blade_mass, calculate the blade_cost. This procudes the same
# result as if we had calculated blade_cost using the rotor_diameter (347627.8)
model_blade_cost = model.calculate("blade_cost", blade_mass=model_blade_mass)
print("model_blade_cost", model_blade_cost)

model_blade_mass 21804.141589129507
model_blade_cost 347627.7901838095


In [13]:
# Lets focus on a different parameter, the nacelle_mass. Currently we don't know what
# inputs are needed to calculate it, but let's give it a go with what we have
print(model.calculate("nacelle_mass", inputs))

KeyError: 'num_bearings is not defined by a function in model Empirical2020. It must be provided as an input.'

In [14]:
# We get a somewhat cryptic error message, something about num_bearings. The problem is
# that num_bearings is an input parameter that is required to calculate nacelle_mass,
# but we have not provided it.

In [ ]:
# Let's be more careful and check if it is possible to calculate the nacelle_mass given
# what we know.

# There may be many nested function calls required to calculate a particular parameter,
# and it is not always obvious if we have all the information we need to calculate it.
# We can use a handy function to determine whether or not a parameter is calculatable
# given what our known inputs are
print(model.is_calculable_with_inputs("nacelle_mass", ["rotor_diameter"]))

In [14]:
# Ok, so we are missing some inputs, but what are they?
print(model.required_inputs_to_calculate("nacelle_mass"))

['num_bearings', 'rotor_diameter', 'rotor_efficiency_max', 'tip_speed_max', 'turbine_rating_MW']


In [15]:
# Our error message was about num_bearings, and that is the first one in the list of
# missing inputs

# Lets make up some values
inputs = {
    "num_bearings": 2,
    "rotor_diameter": 162,
    "rotor_efficiency_max": 0.9,
    "tip_speed_max": 100,
    "turbine_rating_MW": 6.2,
}

# Now we should be able to calculate the nacelle_mass. Let's first check if it is
# calculatable, then perform the calculation
print(model.is_calculable_with_inputs("nacelle_mass", inputs.keys()))
print(model.calculate("nacelle_mass", inputs))

True
176706.7541057522


In [16]:
# Finally, if we know we want to calculate all the parameters in a model, we can use the
# calculate_all method. This is slightly faster than simplly looping over all the
# parameters and calculating each one individually, because it remembers the results
# from the intermediate calculation steps

# Since the calculate_all method is guarenteed to require all the input parameters,
# let's double check we have them all
missing_inputs = set(model.get_inputs()).difference(inputs)
print(missing_inputs)

{'BOS_cost', 'num_blades', 'hub_height'}


In [17]:
# We are still missing a few, let's add them to our existing input dict
inputs |= {
    "hub_height": 150,
    "BOS_cost": 0,
    "num_blades": 3,
}

In [18]:
# Now we have everything we need to calculate all the outputs from the model
# Again we can use the dict itself, or unpack it
outputs = model.calculate_all(inputs)
outputs = model.calculate_all(**inputs)

print(outputs)

{'num_bearings': 2, 'rotor_diameter': 162, 'rotor_efficiency_max': 0.9, 'tip_speed_max': 100, 'turbine_rating_MW': 6.2, 'hub_height': 150, 'BOS_cost': 0, 'num_blades': 3, 'bedplate_mass': 51470.56, 'bedplate_cost': 162996.96940799998, 'rotor_radius': 81.0, 'blade_mass': 21804.141589129507, 'blade_cost': 347627.7901838095, 'blade_surface_area': 197.5, 'blade_transport_cost': 112939.54570000005, 'braking_system_mass': 1232.655, 'braking_system_cost': 9153.202968, 'controls_cost': 143193.96, 'railing_platform_mass': 257.3528, 'crane_mass': 257.3528, 'crane_cost': 1124.1170304000002, 'electrical_connection_cost': 283341.24000000005, 'rotor_angular_velocity_max': 1.2345679012345678, 'rotor_torque_max': 5580000.0, 'gearbox_mass': 45127.71468348047, 'gearbox_cost': 635705.0912032528, 'generator_mass': 14378.400000000001, 'generator_cost': 194695.03872000004, 'hub_mass': 52591.98398997124, 'hub_cost': 223978.7414164895, 'hub_surface_area': 16.0, 'hub_transport_cost': 57591.98398997124, 'hvac_m

In [19]:
# The outputs dict contains all the parameters (not just the outputs, but the inputs
# too). Although most values will be simple (floats or ints), there is no reason your
# model can't return more complex types. For example, here we have a dataframe of
# information about tower sections

outputs["tower_section_data"]

,height,mass,surface_area
tower_section_id,,,
1,25.0,94931.741318,107.5
2,25.0,87337.202012,107.5
3,25.0,79742.662707,107.5
4,25.0,72148.123401,107.5
5,25.0,64553.584096,107.5
6,25.0,56959.044791,107.5


In [20]:
# If we specify only a subset of the required inputs, the calculate_all method will only
# calculate the parameters that are calculatable with what we provide, and will silently
# skip the rest

# Let's see how many parameters we can calculate knowing only the rotor_diameter
model.calculate_all(rotor_diameter=162)

{'rotor_diameter': 162,
 'bedplate_mass': 51470.56,
 'bedplate_cost': 162996.96940799998,
 'rotor_radius': 81.0,
 'blade_mass': 21804.141589129507,
 'blade_cost': 347627.7901838095,
 'blade_surface_area': 197.5,
 'blade_transport_cost': 112939.54570000005,
 'railing_platform_mass': 257.3528,
 'crane_mass': 257.3528,
 'crane_cost': 1124.1170304000002,
 'hub_mass': 52591.98398997124,
 'hub_cost': 223978.7414164895,
 'hub_surface_area': 16.0,
 'hub_transport_cost': 57591.98398997124,
 'hvac_mass': 221.0,
 'hvac_cost': 29925.167999999998,
 'low_speed_shaft_mass': 20191.8064,
 'low_speed_shaft_cost': 262388.48580672,
 'yaw_system_mass': 10589.55901713559,
 'yaw_system_cost': 95979.52710771014,
 'railing_platform_cost': 4805.60030496,
 'nose_cone_mass': 581.381,
 'nose_cone_cost': 7047.0353772,
 'swept_area': 20611.989400202634}

In [21]:
# Finally, if you want to run lots of combinations of parameter values it will be easier
# to use the configuration file. Using the default config file location is as simple as:
from csm import run_config

outputs_config_file = run_config()

Replicate2015Excel: 100%|██████████| 3240/3240 [00:00<00:00, 10356.67it/s]


In [22]:
# When running the models in this way, we get a dataframe of parameter values
empirical_2020_outputs = outputs_config_file[0][1]
scalar_outputs = empirical_2020_outputs[0]
scalar_outputs.head()

,scenario,tip_speed_max,rotor_diameter,turbine_rating_MW,num_bearings,num_blades,BOS_cost,rotor_efficiency_max,hub_height,bedplate_mass,...,parts_shipped_loose_cost,pitch_system_cost,rotor_angular_velocity_max_rpm,rotor_cost,tower_cost,turbine_cost,tower_transport_cost,transport_cost,system_cost,system_specific_cost_per_kW
scenario_number,,,,,,,,,,,,,,,,,,,,,
0,ppinchuck_A,80.0,162.0,6.2,2.0,3.0,0.0,0.9,150.0,51470.56,...,19263.564749,300623.093538,9.431404,1.574532e+06,1.443023e+06,5.329444e+06,204498.0,7.839722e+05,6.113416e+06,986.034792
1,ppinchuck_A,80.0,162.0,7.2,2.0,3.0,0.0,0.9,150.0,51470.56,...,19527.049477,300623.093538,9.431404,1.574532e+06,1.443023e+06,5.546687e+06,204498.0,7.932357e+05,6.339922e+06,880.544749
2,ppinchuck_A,80.0,172.0,6.2,2.0,3.0,0.0,0.9,150.0,58849.36,...,21653.471476,330948.841252,8.883067,1.758761e+06,1.632428e+06,5.843847e+06,238581.0,9.383079e+05,6.782155e+06,1093.895962
3,ppinchuck_A,80.0,172.0,7.2,2.0,3.0,0.0,0.9,150.0,58849.36,...,21922.361575,330948.841252,8.883067,1.758761e+06,1.632428e+06,6.064136e+06,238581.0,9.475768e+05,7.011713e+06,973.848976
4,ppinchuck_A,80.0,182.0,6.2,2.0,3.0,0.0,0.9,150.0,66228.16,...,24179.796643,362659.661863,8.394986,1.951392e+06,1.833175e+06,6.387713e+06,272664.0,1.113081e+06,7.500794e+06,1209.805426


In [23]:
# For our tower_section_data parameter that itself was a dataframe, this gets an
# additional index level to allow it to be joined back to the scalar parameter dataframe
dataframe_outputs = empirical_2020_outputs[1]
dataframe_outputs["tower_section_data"].head(10)

height          mass  surface_area
scenario_number tower_section_id                                    
0               1                   25.0  94931.741318         107.5
                2                   25.0  87337.202012         107.5
                3                   25.0  79742.662707         107.5
                4                   25.0  72148.123401         107.5
                5                   25.0  64553.584096         107.5
                6                   25.0  56959.044791         107.5
1               1                   25.0  94931.741318         107.5
                2                   25.0  87337.202012         107.5
                3                   25.0  79742.662707         107.5
                4                   25.0  72148.123401         107.5